# Notebook 31 — MJO Moisture-Mode Auxiliary Latent (physics-informed 2D)
**Project:** ENSO-BSISO SSL — MJO moisture-constraint experiment
**Author:** Jiayi (jh9141@nyu.edu)

Builds the **explainable 2D latent** the nb30 diagnostics pointed to. Backbone = nb15's label-free
**temporal-contrastive SSL** (which already gives a phase ring + strong ENSO), with the **collapse-fixed
recipe** (tau=0.07 + VICReg, no L2) so the 2D plane stays open. The new ingredient is a
**moisture-mode auxiliary head**: a small decoder must predict the **column moisture (q_col)** and the
**OLR** profile from the 2-D latent.

`L = InfoNCE(z_t, z_{t+/-3})  +  lam_var * VICReg(z)  +  lam_aux * [ MSE(q_hat(z), q_col) + MSE(olr_hat(z), OLR) ]`

Because nb30 measured the real MJO as **moisture-mode-leaning** (q_col ~in phase with convection, NOT a
90 deg skeleton quadrature), we do **not** impose any phase lag — we just make the latent *predict* q and
OLR and then **measure** delta_theta in the learned space, comparing to own-RMM and to the plain nb15 SSL.

Inputs: `X_MJO_bp20_90.npy` (SSL input, bandpassed axis), `labels_aligned_mjo_bp20_90.csv`,
nb29 `qcol_mjo_processed.npy` (full axis -> date-aligned to bp), `mjo_rmm_own_pcs.npy`.
Outputs: `MJO/moisture_constraints/results/aux2d/`.

---

## Cell 1 — Setup + config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, copy
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

PROJECT_DIR = '/content/drive/MyDrive/BSISO_SSL_Project'
MJO_DIR     = f'{PROJECT_DIR}/MJO'
PROC        = f'{MJO_DIR}/data/processed'
MOIST       = f'{MJO_DIR}/moisture_constraints/data/processed'
OUT         = f'{MJO_DIR}/moisture_constraints/results/aux2d'
os.makedirs(OUT, exist_ok=True)

# --- config (collapse-fixed recipe + auxiliary) ---
EMBEDDING_DIM = 2
TEMPERATURE   = 0.07          # collapse fix (nb07d/nb07e); 0.5 collapses 2D onto a line
EPOCHS        = 80
BATCH_SIZE    = 256
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
MAX_DELTA     = 3             # positive pair window [d-3,d+3]\{d}, same year (nb15)
LAM_VAR       = 5.0           # VICReg variance floor (anti-collapse)
LAM_AUX       = 1.0           # moisture/OLR auxiliary weight
SEED          = 42
VAL_STRIDE    = 5
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, ' tau', TEMPERATURE, ' lam_var', LAM_VAR, ' lam_aux', LAM_AUX)

## Cell 2 — Load SSL input + build aligned q_col / OLR targets + year split

`X_bp` is the bandpassed SSL input (channels u850, OLR, u200). The auxiliary targets are the
**column moisture** `q_col` (nb29, date-aligned from the full axis onto the bp axis) and the **OLR**
profile (channel 1 of `X_bp`). Days whose q_col is missing (partial moisture) are excluded from the
auxiliary loss only.

In [ ]:
X_bp   = np.load(f'{PROC}/X_MJO_bp20_90.npy')                  # (M,3,1,180)
labels = pd.read_csv(f'{PROC}/labels_aligned_mjo_bp20_90.csv', parse_dates=['date'])
lons   = np.load(f'{PROC}/longitudes_mjo.npy')
M = len(X_bp); assert len(labels)==M
bpdates = pd.DatetimeIndex(labels['date']).normalize()

# q_col target on the bp axis (from nb29 full-axis processed array)
qcol_full = np.load(f'{MOIST}/qcol_mjo_processed.npy')          # (N,180), NaN where moisture missing
full = pd.read_csv(f'{PROC}/labels_aligned_mjo.csv', parse_dates=['date'])
full_row = {d:i for i,d in enumerate(pd.DatetimeIndex(full['date']).normalize())}
fi = np.array([full_row.get(d,-1) for d in bpdates])
qcol_bp = np.full((M, len(lons)), np.nan, np.float32)
ok = fi>=0; qcol_bp[ok] = qcol_full[fi[ok]]
olr_bp = X_bp[:,1,0,:].astype(np.float32)                      # OLR' target (convection = -olr)
q_finite = np.isfinite(qcol_bp).all(axis=1)
qcol_bp = np.nan_to_num(qcol_bp, nan=0.0).astype(np.float32)   # zeros where missing (masked in loss)
print(f'M={M}  q_col target covered for {int(q_finite.sum())}/{M} bp days')

# own-RMM angle on the bp axis (for comparison)
pcs_full = np.load(f'{PROC}/mjo_rmm_own_pcs.npy')
pcs_bp = np.where(fi[:,None]>=0, pcs_full[np.clip(fi,0,None)], np.nan)
theta_own_bp = np.arctan2(pcs_bp[:,1], pcs_bp[:,0])

years = bpdates.year.values
val_years = sorted(np.unique(years))[::VAL_STRIDE]
is_val = np.isin(years, val_years)
train_idx = np.where(~is_val)[0]; val_idx = np.where(is_val)[0]
print(f'train {len(train_idx)}  val {len(val_idx)}  (val years {val_years[:4]}...)')

## Cell 3 — Encoder (nb15) + decoders + temporal pair sampler + losses

In [ ]:
class MJOEncoderNoL2(nn.Module):
    def __init__(self, embedding_dim=2):
        super().__init__()
        self.conv1=nn.Conv2d(3,16,(1,3),padding=(0,1),bias=False); self.bn1=nn.BatchNorm2d(16); self.pool1=nn.MaxPool2d((1,2))
        self.conv2=nn.Conv2d(16,32,(1,3),padding=(0,1),bias=False); self.bn2=nn.BatchNorm2d(32); self.pool2=nn.MaxPool2d((1,2))
        self.conv3=nn.Conv2d(32,32,(1,3),padding=(0,1),bias=False); self.bn3=nn.BatchNorm2d(32)
        self.gp=nn.AdaptiveAvgPool2d(1); self.fc=nn.Linear(32,embedding_dim)
    def forward(self,x):
        x=self.pool1(F.relu(self.bn1(self.conv1(x))))
        x=self.pool2(F.relu(self.bn2(self.conv2(x))))
        x=F.relu(self.bn3(self.conv3(x)))
        return self.fc(self.gp(x).view(x.size(0),-1))

class FieldDecoder(nn.Module):           # 2D latent -> (180,) field profile
    def __init__(self, d=2, nlon=180):
        super().__init__(); self.net=nn.Sequential(nn.Linear(d,64), nn.ReLU(True), nn.Linear(64,nlon))
    def forward(self,z): return self.net(z)

class TemporalPairSampler:
    def __init__(self, labels_df, allowed, max_delta=3):
        self.labels=labels_df; self.max_delta=max_delta
        sub=labels_df.loc[list(allowed)]
        self.date_to_idx={pd.Timestamp(d):int(i) for d,i in zip(sub['date'].values, sub.index.values)}
    def positive(self, a):
        ad=self.labels.loc[a,'date']; cands=[]
        for dl in range(-self.max_delta,self.max_delta+1):
            if dl==0: continue
            t=ad+pd.Timedelta(days=dl)
            if t.year!=ad.year: continue
            k=pd.Timestamp(t)
            if k in self.date_to_idx: cands.append(self.date_to_idx[k])
        return (a, int(np.random.choice(cands))) if cands else (a,a)

class AuxDataset(Dataset):
    def __init__(self, X, q, qfin, labels_df, idx, max_delta=3):
        self.X=X; self.q=q; self.qfin=qfin; self.idx=np.asarray(idx)
        self.s=TemporalPairSampler(labels_df, idx, max_delta)
    def __len__(self): return len(self.idx)
    def __getitem__(self,i):
        a=int(self.idx[i]); a,b=self.s.positive(a)
        return (torch.from_numpy(self.X[a]).float(), torch.from_numpy(self.X[b]).float(),
                torch.from_numpy(self.q[a]).float(), torch.tensor(float(self.qfin[a])))

def infonce_raw(zA,zB,tau):
    sim=torch.matmul(zA,zB.T)/tau
    return F.cross_entropy(sim, torch.arange(zA.size(0),device=zA.device))

def vicreg_var(z, gamma=1.0, eps=1e-4):
    std=torch.sqrt(z.var(0)+eps); return torch.mean(F.relu(gamma-std))

train_loader=DataLoader(AuxDataset(X_bp,qcol_bp,q_finite,labels,train_idx,MAX_DELTA),
                        batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=(device.type=='cuda'),drop_last=True)
print('batches/epoch:', len(train_loader))

## Cell 4 — Train (InfoNCE + VICReg + moisture/OLR auxiliary)

In [ ]:
encoder=MJOEncoderNoL2(EMBEDDING_DIM).to(device)
qdec=FieldDecoder(EMBEDDING_DIM,len(lons)).to(device)
odec=FieldDecoder(EMBEDDING_DIM,len(lons)).to(device)
params=list(encoder.parameters())+list(qdec.parameters())+list(odec.parameters())
opt=optim.Adam(params,lr=LR,weight_decay=WEIGHT_DECAY)
sch=optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS,eta_min=1e-5)

hist={'loss':[], 'nce':[], 'aux':[], 'var':[]}
t0=time.time()
for ep in range(EPOCHS):
    encoder.train(); qdec.train(); odec.train()
    agg={'loss':0,'nce':0,'aux':0,'var':0}; nb=0
    for xa,xb,qa,qf in train_loader:
        xa,xb,qa,qf=xa.to(device),xb.to(device),qa.to(device),qf.to(device)
        za=encoder(xa); zb=encoder(xb)
        L_nce=infonce_raw(za,zb,TEMPERATURE)
        L_var=vicreg_var(za)+vicreg_var(zb)
        qhat=qdec(za); ohat=odec(za); otgt=xa[:,1,0,:]
        w=qf.clamp(min=0); denom=w.sum().clamp(min=1)
        L_q=((qhat-qa).pow(2).mean(1)*w).sum()/denom          # masked: only days with q target
        L_o=F.mse_loss(ohat,otgt)
        L_aux=L_q+L_o
        L=L_nce+LAM_VAR*L_var+LAM_AUX*L_aux
        opt.zero_grad(); L.backward(); opt.step()
        agg['loss']+=L.item(); agg['nce']+=L_nce.item(); agg['aux']+=L_aux.item(); agg['var']+=L_var.item(); nb+=1
    sch.step()
    for k in agg: hist[k].append(agg[k]/nb)
    if (ep+1)%10==0 or ep<3:
        print(f'ep {ep+1:3d}/{EPOCHS}  loss={hist["loss"][-1]:.3f}  nce={hist["nce"][-1]:.3f}  '
              f'aux={hist["aux"][-1]:.3f}  var={hist["var"][-1]:.3f}  lr={sch.get_last_lr()[0]:.1e}')
torch.save(encoder.state_dict(), f'{OUT}/encoder_aux2d.pth')
json.dump(hist, open(f'{OUT}/training_history.json','w'))
print(f'Done in {(time.time()-t0)/60:.1f} min')

## Cell 5 — Extract embeddings + collapse check

In [ ]:
encoder.eval()
Z=np.zeros((M,EMBEDDING_DIM),np.float32)
with torch.no_grad():
    for s in range(0,M,256):
        Z[s:s+256]=encoder(torch.from_numpy(X_bp[s:s+256]).float().to(device)).cpu().numpy()
np.save(f'{OUT}/embeddings.npy', Z)
ev=np.linalg.eigvalsh(np.cov(Z.T)); ev=np.sort(ev)[::-1]
eff_rank=float((ev.sum()**2)/(ev**2).sum())
print(f'embeddings {Z.shape}  eig {ev.round(3)}  eff_rank={eff_rank:.2f}  (want ~2; ~1 = collapsed)')

## Cell 6 — Diagnostics in the LEARNED latent: delta_theta, ENSO, vs own-RMM / nb15

Measure the moisture-convection phase offset in this physics-informed latent and compare to the own-RMM
clock and to the plain nb15 SSL embedding. Also the ENSO displacement z-score and the 2-D plane.

In [ ]:
REGIONS={'IndianOcean':(60,90),'MaritimeContinent':(100,130),'WestPacific':(140,170)}
rmask={k:(lons>=v[0])&(lons<=v[1]) for k,v in REGIONS.items()}
phase=labels['phase'].values.astype(int); amp=labels['amplitude'].values
enso=labels['enso_category'].values; weak=labels['weak_mjo'].values.astype(bool)
active=(~weak)&(amp>=1.0)&q_finite
conv=-olr_bp
ph_ang=(phase-1)/8.0*2*np.pi
def circ_corr(a,b):
    a=a-np.angle(np.mean(np.exp(1j*a))); b=b-np.angle(np.mean(np.exp(1j*b)))
    return float(np.sum(np.sin(a)*np.sin(b))/np.sqrt(np.sum(np.sin(a)**2)*np.sum(np.sin(b)**2)+1e-12))
def wrapdeg(a): return np.degrees((a+np.pi)%(2*np.pi)-np.pi)
def offsets(theta, mask):
    if circ_corr(theta[mask], ph_ang[mask])<0: theta=-theta
    e=np.exp(1j*theta[mask]); out={}
    for rk,rm in rmask.items():
        Af=np.nanmean(qcol_bp[mask]*e[:,None],0)[rm].sum(); Ac=np.nanmean(conv[mask]*e[:,None],0)[rm].sum()
        out[rk]=round(wrapdeg(np.angle(Af)-np.angle(Ac)),1)
    return out, round(circ_corr(theta[mask], ph_ang[mask]),2)

theta_z=np.arctan2(Z[:,1],Z[:,0])
oz,cz=offsets(theta_z,active)
oo,co=offsets(theta_own_bp,active)
nb15p=f'{MJO_DIR}/results/ssl/embeddings.npy'
print('delta_theta(q_col,conv) in regions (deg):')
print(f'  aux2d (learned):  {oz}   circ_corr(theta,phase)={cz}')
print(f'  own-RMM:          {oo}   circ_corr={co}')
if os.path.exists(nb15p):
    Z15=np.load(nb15p)
    if len(Z15)==M:
        o15,c15=offsets(np.arctan2(Z15[:,1],Z15[:,0]),active); print(f'  nb15 SSL:         {o15}   circ_corr={c15}')

# ENSO displacement z-score in the learned latent
def enso_z(emb):
    idx=np.where(active)[0]; e=emb[idx]; lab=labels.iloc[idx].reset_index(drop=True); obs=[]
    for p in range(1,9):
        mEN=(lab['phase']==p)&(lab['enso_category']=='El Nino'); mLN=(lab['phase']==p)&(lab['enso_category']=='La Nina')
        if mEN.sum()<3 or mLN.sum()<3: continue
        obs.append(np.linalg.norm(e[mEN.values].mean(0)-e[mLN.values].mean(0)))
    rng=np.random.default_rng(0); base=[]
    for _ in range(200):
        sh=lab['enso_category'].sample(frac=1,random_state=rng.integers(1e6)).values; t=[]
        for p in range(1,9):
            mph=(lab['phase']==p).values; mEN=mph&(sh=='El Nino'); mLN=mph&(sh=='La Nina')
            if mEN.sum()<3 or mLN.sum()<3: continue
            t.append(np.linalg.norm(e[mEN].mean(0)-e[mLN].mean(0)))
        if t: base.append(np.mean(t))
    return float((np.mean(obs)-np.mean(base))/(np.std(base)+1e-8))
print(f'\nENSO displacement z (aux2d latent): {enso_z(Z):.2f}   (nb15 SSL ~14-18, supervised ~2.5)')

# 2-D plane viz
sub=np.where(active)[0]; sub=np.random.default_rng(0).choice(sub,min(5000,len(sub)),replace=False)
fig,ax=plt.subplots(1,3,figsize=(18,5.2))
hsv=plt.cm.hsv(np.linspace(0,1,9)); epal={'El Nino':'#d62728','Neutral':'#7f7f7f','La Nina':'#1f77b4'}
for p in range(1,9):
    m=sub[phase[sub]==p]; ax[0].scatter(Z[m,0],Z[m,1],s=6,alpha=.5,color=hsv[p-1],label=f'P{p}')
ax[0].set_title('by RMM phase'); ax[0].legend(fontsize=6,ncol=2)
for ccat in epal:
    m=sub[enso[sub]==ccat]; ax[1].scatter(Z[m,0],Z[m,1],s=6,alpha=.5,color=epal[ccat],label=ccat)
ax[1].set_title('by ENSO'); ax[1].legend(fontsize=7)
scA=ax[2].scatter(Z[sub,0],Z[sub,1],s=6,alpha=.6,c=amp[sub],cmap='viridis'); ax[2].set_title('by amplitude'); plt.colorbar(scA,ax=ax[2])
for a in ax: a.set_xticks([]); a.set_yticks([])
fig.suptitle(f'Physics-informed aux2d latent (eff_rank {eff_rank:.2f}, ENSO z {enso_z(Z):.1f})',fontweight='bold')
plt.tight_layout(); p=f'{OUT}/aux2d_latent.png'; plt.savefig(p,dpi=130,bbox_inches='tight'); plt.show(); print('Saved',p)

summary={'eff_rank':round(eff_rank,2),'enso_z':round(enso_z(Z),2),
         'delta_theta_aux2d':oz,'delta_theta_own_rmm':oo,'circ_corr_aux2d':cz}
json.dump(summary, open(f'{OUT}/aux2d_summary.json','w'), indent=2)
print('Saved', f'{OUT}/aux2d_summary.json'); print(summary)

---
## Done!
Outputs in `MJO/moisture_constraints/results/aux2d/`: `encoder_aux2d.pth`, `embeddings.npy`,
`aux2d_latent.png`, `aux2d_summary.json`.

**Read it:** if the learned latent keeps a clean phase ring (eff_rank ~2, ENSO z high) AND its
`delta_theta(q_col, conv)` matches own-RMM's moisture-mode lead, the auxiliary made a physically
interpretable MJO plane where angle = moisture/convection cycle and radius/sector = ENSO. If interpretable
delta_theta improves over plain nb15, report it; if phase skill drops, report the tradeoff.

**Next:** ENSO-stratified delta_theta in this latent; optionally a measured-lag regularizer (only if the
data supports it) or escalate to 3-D (2-D phase plane + slow ENSO coordinate).

---
*DDCS Project | jh9141@nyu.edu*